# FASE EXPLORATORIO FOOD DELIVERY

En esta primera etapa exploratoria del proceso ETL se realizó un análisis inicial de los datos relacionados con la data restaurant-menu.csv y restaurants.csv, con el objetivo de comprender la estructura, caracteristicas y calidad de la data antes de aplicar los procesos de extraccion, transformacion y carga.

Durante la exploración se identificaron las fuentes de datos. Entre los datos evaluados se encontro informacion de nombre de restaurante, score, ratings, rango de precios, direccion completa, producto, precio, etc.

Las principales objetivos de esta fase fueron:

* Revisar la estructura de los archivos o bases de datos de origen.
* Identificar las tablas, columnas y relaciones existentes entre los datos.
* Analizar tipos de datos y formatos de las variables.
* Detectar valores nulos, duplicados o inconsistencias.
* Evaluar la calidad y disponibilidad de la información.
* Definir posibles reglas de limpieza y transformación para las siguientes etapas del ETL.

In [46]:
## clonar o actualizar repositorio github en colab

from pathlib import Path
import os

ruta_proyecto = Path("/content/etl-python-analisis-food-delivery")

# Si el repositorio no existe, lo clona
if not ruta_proyecto.exists():
    %cd /content
    !git clone https://github.com/dannyturpo-data/etl-python-analisis-food-delivery.git

# Si ya existe, actualiza cambios
else:
    %cd /content/etl-python-analisis-food-delivery
    !git pull origin main

/content/etl-python-analisis-food-delivery
From https://github.com/dannyturpo-data/etl-python-analisis-food-delivery
 * branch            main       -> FETCH_HEAD
Already up to date.


In [30]:
## configuracion

import sys

if str(ruta_proyecto) not in sys.path:
    sys.path.append(str(ruta_proyecto))

## importando librerias

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display

## funciones propias

from src.exploracion_df import exploratorio_df

## PARTE 1: EXTRACCION Y DIAGNOSTICOS (EXTRACT)

### ANALISIS DATA RESTAURANTS.csv

In [31]:
## dataframe df_restaurantes

df_restaurantes = pd.read_csv('/content/etl-python-analisis-food-delivery/restaurants.csv/restaurants.csv')

In [32]:
exploratorio_df(df_restaurantes)

RESUMEN GENERAL

MUESTRA ALEATORIA


,id,position,name,score,ratings,category,price_range,full_address,zip_code,lat,lng
9912,9913,16,Cosmic Wings - Wenatchee,3.8,10.0,"American, Bar Food, Wings, Fast Food, Chicken,...",$,"1300A N Miller St, Wenatchee, WA, 98801",98801,47.440200,-120.325060
59957,59958,243,Subway (9410 Walnut St),4.9,14.0,"Fast Food, Sandwich, American",$,"9410 Walnut St, Dallas, TX, 75243",75243,32.923490,-96.736064
24792,24793,49,KFC (10310 Iron Bridge Road),4.3,19.0,"American, Fast Food, wings, Family Meals",$,"10310 Iron Bridge Road, Chesterfield, VA, 23831",23831,37.372233,-77.502107
52562,52563,113,Mia Bella Trattoria,4.5,85.0,"Italian, Pasta, Comfort Food",NaN,"3773 Richmond Ave, Houston, TX, 77046",77046,29.732418,-95.437250
38983,38984,5,Nekter Juice Bar (Colleyville),4.7,57.0,"Juice and Smoothies, Breakfast and Brunch, Veg...",$,"Nekter Juice Bar 5615, Colleyville, TX, 76034",76034,32.891068,-97.147861




FILAS      : 63,469
COLUMNAS   : 11
DUPLICADOS : 0


TIPOS DE DATOS


,id,position,name,score,ratings,category,price_range,full_address,zip_code,lat,lng
0,int64,int64,object,float64,float64,object,object,object,object,float64,float64





CANTIDAD DE VALORES NULOS POR COLUMNA


,id,position,name,score,ratings,category,price_range,full_address,zip_code,lat,lng
Valores nulos,0,0,0,28167,28167,85,10617,453,517,0,0





PORCENTAJE DE VALORES NULOS POR COLUMNA


,id,position,name,score,ratings,category,price_range,full_address,zip_code,lat,lng
% Valores nulos,0.0,0.0,0.0,44.38,44.38,0.13,16.73,0.71,0.81,0.0,0.0





CANTIDAD DE VALORES ÚNICOS POR COLUMNA


,id,position,name,score,ratings,category,price_range,full_address,zip_code,lat,lng
Valores únicos,63469,300,60865,33,433,15574,5,54946,3082,57227,57249


63,469 filas y 11 columnas indican un dataset de tamaño mediano, la cantidad de registros es considerable, por lo que los datos faltantes pueden representar miles de observaciones.

No existen registros completamente repetidos. Esto es positivo porque no será necesario eliminar duplicados exactos antes del análisis. Sin embargo, esto no garantiza que no existan duplicados lógicos.

price_range debe revisarse, contiene valores categoricos. Contiene solo 5 valores unicos, es decir es una variable categorica.

Existen variables con valores nulos, este es el principal problema del dataset, casi la mitad de los negocios no tiene informacion de puntuacion ni cantidad de reseñas. Quizas sean negocios nuevos, con poca presencia, datos incompletos de la fuente original. Se debe tener cuidado con esta data faltante, se considera no eliminar estos registros ya que se perderian casi la mitad del dataset por efecto de algunas variables.

La variable full_address contiene alrededor del 16.73% de valores nulos, es una perdida importante porque la direccion puede ser necesaria para analisis geografico, agrupacion por zonas, mapas, etc. Pero como existen informacion de lat y lon podria reconstruirse parcialmente la ubicacion.

Variable category contiene alrededor de 15k valores unicos, una cantidad muy alta de categorias diferentes, seria necesario normalizar categorias, agrupar categorias similares.

Variable name tiene un 95.9% de valores unicos, casi la mayoria de negocios tiene nombres diferentes, esperable en un data set de este tipo.


### ANALISIS DATA RESTAURANT_MENUS.csv

In [38]:
# creando diccionarios de DataFrames, se guardan dentro de dfs

ruta_csv = Path("/content/etl-python-analisis-food-delivery/restaurant-menus.csv")

dfs = {}

# busca todos los archivos csv dentro de la ruta
for archivo in sorted(ruta_csv.glob("*.csv")):
    # guarda dentro del diccionario
    # archivo.stem archivo son extension
    dfs[archivo.stem] = pd.read_csv(archivo)

In [41]:
# que DataFrame se crearon
dfs.keys()

dict_keys(['restaurant_menus_parte_1', 'restaurant_menus_parte_10', 'restaurant_menus_parte_2', 'restaurant_menus_parte_3', 'restaurant_menus_parte_4', 'restaurant_menus_parte_5', 'restaurant_menus_parte_6', 'restaurant_menus_parte_7', 'restaurant_menus_parte_8', 'restaurant_menus_parte_9'])

In [42]:
# cuantos archivos se cargo
len(dfs)

10

In [43]:
# tamaño de cada DataFrame

for nombre, df in dfs.items():
    print(nombre, df.shape)

restaurant_menus_parte_1 (83635, 5)
restaurant_menus_parte_10 (83635, 5)
restaurant_menus_parte_2 (83635, 5)
restaurant_menus_parte_3 (83635, 5)
restaurant_menus_parte_4 (83635, 5)
restaurant_menus_parte_5 (83635, 5)
restaurant_menus_parte_6 (83635, 5)
restaurant_menus_parte_7 (83635, 5)
restaurant_menus_parte_8 (83635, 5)
restaurant_menus_parte_9 (83635, 5)


In [45]:
# analisis de cada df

for nombre, df in dfs.items():
    print("="*100)
    print(nombre)
    exploratorio_df(df)

restaurant_menus_parte_1
RESUMEN GENERAL

MUESTRA ALEATORIA


,restaurant_id,category,name,description,price
18828,276,Non-Alcoholic Beverages,Flavored Lemonades,Bright and refreshing lemonade in your choice ...,3.09 USD
37176,510,Condiments,Hot Picante Salsa,Limit of 2,0.0 USD
10049,155,Kid's Menu,Kid's Grilled Chicken Sandwich,NaN,8.4 USD
7844,126,Combos,Urban Cheeseburger Combo,"All Beef Patty, Topped with Cheese, Lettuce, T...",12.99 USD
20266,301,Kids' Meals,Wacky Pack® Jr Burger,"A juicy, 100% pure beef patty, and crinkle-cu...",0.0 USD




FILAS      : 83,635
COLUMNAS   : 5
DUPLICADOS : 105


TIPOS DE DATOS


,restaurant_id,category,name,description,price
0,int64,object,object,object,object





CANTIDAD DE VALORES NULOS POR COLUMNA


...

,restaurant_id,category,name,description,price
Valores nulos,0,0,0,28304,0





PORCENTAJE DE VALORES NULOS POR COLUMNA


,restaurant_id,category,name,description,price
% Valores nulos,0.0,0.0,0.0,33.84,0.0





CANTIDAD DE VALORES ÚNICOS POR COLUMNA


,restaurant_id,category,name,description,price
Valores únicos,1174,4090,46533,36329,2130


Contando las 10 partes del dataset, es un dataset relativamente grande, tiene pocas variables. Un restaurante puede tener multiples registros.

La cantidad de duplicados es baja, no representa un problema critico, deben investigarse antes de ser eliminados.

La variable precio deberia ser numerica, pero esta almacenada como texto.

Los valores faltantes representan alrededor del 20%, pero estan concentrados en su totalidad en la variable description, esta variable puede considerarse una variable complementaria, no critica.

La variable category contiene muchos valores unicos, por lo que puede contener categorias inconsistentes (pizza, pizzas, etc). Se requiere una normalizacion pasar todo a minusculas, eliminar espacios, corregir caracteres especiales, etc.